<a href="https://colab.research.google.com/github/sudeepbhattad02-creator/AIProject2/blob/main/UTAIGA_Project_2_Full_Code_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Problem Statement

### Business Context

GlobalEdge Brokerage is a mid-sized brokerage firm operating across multiple countries, serving thousands of retail and institutional clients through a network of equity brokers. Brokers handle investment recommendations, portfolio reviews, and market advisory discussions across equity, forex, commodity, crypto, and international stock markets. Each broker manages hundreds of clients and is expected to stay updated with overnight market developments before the market opens.

Every morning, brokers must review large volumes of financial news, stock-price movements, earnings discussions, analyst commentary, and SEC regulatory filings within a very limited preparation window. In practice, brokers can only read a small portion of the available information before their first client calls begin. As a result, many important signals, disclosures, and market events remain unnoticed.

This creates an intelligence gap during client interactions. Brokers often rely on partial information, memory, or fragmented market sources while answering client questions. Important disclosures buried inside long 10-K and 10-Q filings are difficult to review manually, and brokers may struggle to provide evidence-backed responses when clients ask for justification or supporting references. This increases both compliance risk and trust-related challenges.

To solve this problem, GlobalEdge wants to build a financial intelligence assistant that allows brokers to ask natural-language questions and receive grounded answers backed by real financial data, news articles, and regulatory filings. The system should reduce information overload, improve market coverage, and help brokers make faster and more confident advisory decisions without adding technical complexity to their workflow.

### Objective

This project proposes a proof-of-concept Retrieval-Augmented Generation (RAG) financial intelligence system that combines financial news, stock-price data, and SEC filings into a searchable intelligence layer. The system uses semantic retrieval with a local Chroma vector database to fetch relevant financial context and generate grounded answers for broker questions using large language models. Brokers can ask plain-English questions such as market sentiment analysis, company-risk queries, filing-related questions, or cross-market comparisons and receive evidence-backed responses.

To improve the quality and reliability of generated answers, the system introduces a DeepEval-based prompt optimization workflow using GEPA (Genetic-Pareto Prompt Optimization). Instead of manually refining prompts, the workflow uses benchmark financial question-answer examples and evaluation metrics such as faithfulness, groundedness, relevance, and actionability to optimize the final answering prompt.

The project also evaluates multiple RAG configurations by tuning retrieval and generation parameters such as chunk size, chunk overlap, number of retrieved chunks, temperature, top-p, and maximum token limits. Each configuration is evaluated using DeepEval metrics to identify the most effective combination for grounded financial reasoning and broker-style question answering.

The final system uses the optimized prompt together with the best-performing RAG configuration to generate accurate, grounded, and production-style financial intelligence responses for broker workflows.

### Data Description

The system uses three financial intelligence datasets collected through a custom data ingestion pipeline and stored locally for the RAG workflow.
1. Global Financial News (global_news.csv)
Contains financial news articles with titles, article content, publication dates, URLs, and source metadata. The dataset covers company announcements, market developments, macroeconomic events, sector movements, and sentiment-related signals.
2. Global Stock Prices (all_prices_clean.csv)
Contains historical stock-price data across global equity markets, crypto markets, and forex markets. Each record includes ticker symbols, company names, open/high/low/close prices, trading volume, and timestamps for market analysis and trend evaluation.
3. SEC Regulatory Filings (sec_filings.txt)
Contains regulatory filing documents including 10-K annual reports, 10-Q quarterly reports, and other SEC disclosures. These filings include financial statements, governance information, risk disclosures, operational commentary, and compliance-related information used for grounded financial reasoning.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Installing the necessary libraries with specified versions
%pip install -qU \
    "chromadb==1.5.9" \
    "langchain-community==0.4.1" \
    "langchain-chroma==1.1.0" \
    "langchain-openai==1.2.1" \
    "langchain-text-splitters==1.1.2" \
    "pandas==3.0.3" \
    "numpy==2.4.4" \
    "scikit-learn>=1.4.0" \
    "python-dotenv>=1.0.1" \
    "tqdm>=4.66.0"\
    "deepeval==4.0.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 95.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 98.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.9/954.9 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 115.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for VSCode), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Importing the necessary libraries

# Standard Python libraries for file handling, text processing, JSON handling,
# randomness, timing, and basic utilities.
import os
import re
import json
import random
import time
from pathlib import Path
from collections import Counter
import pandas as pd

# DeepEval libraries for prompt optimization and evaluation.
from deepeval.prompt import Prompt
from deepeval.dataset import Golden
from deepeval.metrics import GEval
from deepeval.optimizer import PromptOptimizer
from deepeval.optimizer.algorithms import GEPA
from deepeval.test_case import LLMTestCase, SingleTurnParams
from deepeval.models import GPTModel
from deepeval.optimizer.policies import TieBreaker

# LangChain libraries for document loading, text splitting, embeddings,
# vector storage, and LLM interaction.
from langchain_community.document_loaders import CSVLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

### Notebook Setup: Dependencies and Environment
This section handles the initial setup for the entire project. It ensures all necessary Python libraries are installed with specific versions and imports them into the environment. Following this, critical environment variables are loaded, and project-specific paths for data, the Chroma vector database, DeepEval artifacts, and prompt optimization outputs are defined and created. This ensures a consistent and ready-to-use environment for all subsequent RAG and evaluation operations.

## Environment Setup and Project Initialization

In [ ]:
from dotenv import load_dotenv
import os
from pathlib import Path

# Load environment variables from .env file
load_dotenv()

# Define paths for local datasets
GLOBAL_NEWS_PATH = "/content/global_news.csv"
STOCK_PRICES_PATH = "/content/stock_price_details.csv"
SEC_FILINGS_PATH = "/content/sec_filings_10k.txt"

# Define directories for Chroma DB, DeepEval artifacts, and prompt optimization
CHROMA_DB_PATH = "./chroma_db"
DEEPEVAL_ARTIFACTS_PATH = "./deepeval_artifacts"
PROMPT_OPTIMIZATION_PATH = "./prompt_optimization"

# Create directories if they don't exist
for path in [CHROMA_DB_PATH, DEEPEVAL_ARTIFACTS_PATH, PROMPT_OPTIMIZATION_PATH]:
    Path(path).mkdir(parents=True, exist_ok=True)

# Configure OpenAI API credentials
# If using Colab, ensure your API key is stored in Colab secrets under 'OPENAI_API_KEY'
# and OPENAI_BASE_URL is 'https://api.openai.com/v1' if not explicitly set.

try:
    from google.colab import userdata
    colab_env = True
except ImportError:
    colab_env = False

# Handle OPENAI_API_KEY
if os.getenv("OPENAI_API_KEY") is None:
    if colab_env:
        api_key_from_secrets = userdata.get("OPENAI_API_KEY")
        if api_key_from_secrets:
            os.environ["OPENAI_API_KEY"] = api_key_from_secrets
        else:
            print("Warning: OPENAI_API_KEY not found in Colab secrets. Please set it.")
    else:
        print("Warning: OPENAI_API_KEY not found in environment variables. Please set it.")

# Handle OPENAI_BASE_URL
if os.getenv("OPENAI_BASE_URL") is None:
    if colab_env:
        base_url_from_secrets = userdata.get("OPENAI_BASE_URL")
        if base_url_from_secrets:
            os.environ["OPENAI_BASE_URL"] = base_url_from_secrets
        else:
            os.environ["OPENAI_BASE_URL"] = "https://api.openai.com/v1"
            print("OPENAI_BASE_URL not found in Colab secrets, defaulting to 'https://api.openai.com/v1'.")
    else:
        os.environ["OPENAI_BASE_URL"] = "https://api.openai.com/v1"
        print("OPENAI_BASE_URL not found in environment variables, defaulting to 'https://api.openai.com/v1'.")

# Define the embedding model and answer generation model
EMBEDDING_MODEL = "text-embedding-ada-002"
ANSWER_GENERATION_MODEL = "gpt-4o-mini"

# Display active project directories and dataset paths for verification
print("--- Dataset Paths ---")
print(f"Global Financial News: {GLOBAL_NEWS_PATH}")
print(f"Global Stock Prices: {STOCK_PRICES_PATH}")
print(f"SEC Regulatory Filings: {SEC_FILINGS_PATH}")

print("\n--- Project Directories ---")
print(f"Chroma DB Path: {CHROMA_DB_PATH}")
print(f"DeepEval Artifacts Path: {DEEPEVAL_ARTIFACTS_PATH}")
print(f"Prompt Optimization Path: {PROMPT_OPTIMIZATION_PATH}")

print("\n--- OpenAI Models ---")
print(f"Embedding Model: {EMBEDDING_MODEL}")
print(f"Answer Generation Model: {ANSWER_GENERATION_MODEL}")


--- Dataset Paths ---
Global Financial News: /content/global_news.csv
Global Stock Prices: /content/stock_price_details.csv
SEC Regulatory Filings: /content/sec_filings_10k.txt

--- Project Directories ---
Chroma DB Path: ./chroma_db
DeepEval Artifacts Path: ./deepeval_artifacts
Prompt Optimization Path: ./prompt_optimization

--- OpenAI Models ---
Embedding Model: text-embedding-ada-002
Answer Generation Model: gpt-4o-mini


### Data Loading and Vector Store Creation for Baseline RAG
This segment focuses on ingesting the various financial data sources—global news, stock prices, and SEC filings—into the RAG pipeline. It utilizes LangChain loaders to process CSV and text files, and then employs a `RecursiveCharacterTextSplitter` to break down large SEC documents into manageable chunks. Finally, an OpenAI embedding model is used to convert all loaded documents and chunks into vector embeddings, which are then stored in a persistent local Chroma vector database. This database will be used for efficient semantic retrieval.

# Section 1: Baseline Retrieval-Augmented Generation (RAG) Pipeline

## 1.1 Load the CSV files with `CSVLoader`

In [ ]:
from langchain_community.document_loaders import CSVLoader

# Initialize CSVLoader for global financial news
news_loader = CSVLoader(file_path=GLOBAL_NEWS_PATH)
news_documents = news_loader.load()

# Initialize CSVLoader for global stock prices
stock_loader = CSVLoader(file_path=STOCK_PRICES_PATH)
stock_documents = stock_loader.load()

# Combine the documents from both CSVs
all_documents = news_documents + stock_documents

print(f"Loaded {len(news_documents)} news documents.")
print(f"Loaded {len(stock_documents)} stock price documents.")
print(f"Total documents loaded: {len(all_documents)}")


Loaded 760 news documents.
Loaded 2986 stock price documents.
Total documents loaded: 3746


## 1.2 Load the SEC filings text and split it into chunks


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize TextLoader for SEC filings
sec_loader = TextLoader(file_path=SEC_FILINGS_PATH)
sec_documents = sec_loader.load()

# Initialize RecursiveCharacterTextSplitter
# These values are common starting points; they might be tuned later during RAG configuration.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,       # Max size of chunks
    chunk_overlap=200,     # Overlap between chunks
    length_function=len,
    is_separator_regex=False,
)

# Split the SEC documents into chunks
sec_document_chunks = text_splitter.split_documents(sec_documents)

print(f"Loaded {len(sec_documents)} SEC filing documents.")
print(f"Split into {len(sec_document_chunks)} chunks.")

# Add the SEC document chunks to the all_documents list
all_documents.extend(sec_document_chunks)

print(f"Total documents (news + stock + SEC chunks): {len(all_documents)}")


Loaded 1 SEC filing documents.
Split into 10065 chunks.
Total documents (news + stock + SEC chunks): 13811


## 1.3 Build a local Chroma vector store

In [ ]:
# Initialize OpenAIEmbeddings
# The embedding model is defined as EMBEDDING_MODEL ('text-embedding-ada-002')
# The API key is sourced from environment variables/Colab secrets
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# Create a Chroma vector store from the combined documents and persist it to disk
vector_store = Chroma.from_documents(
    documents=all_documents,
    embedding=embeddings,
    persist_directory=CHROMA_DB_PATH
)

print(f"Chroma vector store created and persisted to {CHROMA_DB_PATH}")


Chroma vector store created and persisted to ./chroma_db


### Baseline RAG Answer Generation Logic
This part establishes the core logic for the baseline Retrieval-Augmented Generation (RAG) pipeline. It initializes the `ChatOpenAI` model, sets up functions to retrieve relevant documents from the Chroma vector store based on a question, and formats these documents into a concise context. Crucially, it defines the system and user prompts that guide the LLM's behavior, instructing it to generate answers *only* from the provided financial context and maintain a professional tone. These components are then integrated into a single `get_rag_answer` function.

## 1.4 Retrieve Context and Generate Answers using the Baseline RAG Pipeline

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Initialize the ChatOpenAI model for answer generation
# It uses the API key and base URL from the environment variables set in cell 'a464c83e'
llm = ChatOpenAI(
    model=ANSWER_GENERATION_MODEL, # 'gpt-4o-mini' from 'a464c83e'
    temperature=0.0,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL")
)

print(f"Chat model initialized: {ANSWER_GENERATION_MODEL}")

Chat model initialized: gpt-4o-mini


Next, we'll define a function to format the retrieved documents. This function will extract relevant content and metadata (like source, date, and ticker) to create a concise context block for the LLM.

In [ ]:
def retrieve_docs(question: str, k: int = 4):
    """
    Retrieve the most semantically relevant document chunks
    from the Chroma vector database.
    """
    return vector_store.similarity_search(question, k=k)

In [ ]:
def format_docs(docs):
    formatted_string = ""
    for i, doc in enumerate(docs):
        content = doc.page_content.strip()
        metadata = doc.metadata

        # Extract relevant metadata fields
        source = metadata.get('source', 'N/A')
        row = metadata.get('row', 'N/A') # For CSVs
        date = metadata.get('date', 'N/A') # For news/stock
        ticker = metadata.get('ticker', 'N/A') # For stock

        # Adjust source path for display if it's a content file path
        if source.startswith('/content/'):
            source = os.path.basename(source)

        formatted_string += f"-- Document {i+1} --\n"
        formatted_string += f"Source: {source}"
        if row != 'N/A':
            formatted_string += f", Row: {row}"
        if date != 'N/A':
            formatted_string += f", Date: {date}"
        if ticker != 'N/A':
            formatted_string += f", Ticker: {ticker}"
        formatted_string += f"\nContent: {content}\n\n"
    return formatted_string

print("Document formatting function 'format_docs' defined.")

Document formatting function 'format_docs' defined.


Now, we will define the core RAG function. This function will take a broker's question, retrieve relevant documents from the Chroma vector store, format them, and then use the initialized LLM to generate an answer based *only* on the provided context. It will also return the retrieved documents and the formatted context for transparency.

In [ ]:
BASELINE_SYSTEM_PROMPT = """
You are a highly knowledgeable financial intelligence assistant.
Your goal is to answer questions based *only* on the provided financial context.
Do NOT use any outside knowledge. If the answer is not in the context, state that you cannot answer based on the provided information.
Maintain a professional and neutral tone. Prioritize factual accuracy.
Cite the source(s) from the retrieved documents if explicitly asked.
""".strip()

In [ ]:
BASELINE_USER_PROMPT = """
Broker Question:
{question}

Retrieved Context:
{context}
""".strip()

In [ ]:
def get_rag_answer(question: str, k: int = 4):
    """
    Retrieves relevant documents and generates an answer using a RAG pipeline.

    Args:
        question (str): The broker's question.

    Returns:
        tuple: (answer_text, retrieved_docs, formatted_context_str)
    """

    # Retrieve relevant chunks from Chroma
    retrieved_docs = retrieve_docs(question, k=k)

    # Convert retrieved documents into prompt-ready context
    formatted_context_str = format_docs(retrieved_docs)

    # Inject the runtime values into the prompt template
    user_prompt = BASELINE_USER_PROMPT.format(
        question=question,
        context=formatted_context_str,
    )

    # Generate the final response
    response = llm.invoke([
        SystemMessage(content=BASELINE_SYSTEM_PROMPT),
        HumanMessage(content=user_prompt),
    ])

    return response.content, retrieved_docs, formatted_context_str

### Demonstrating Baseline RAG and Setting up Evaluation Metrics
Here, the end-to-end functionality of the baseline RAG pipeline is demonstrated with a sample financial question. Following this qualitative check, the section prepares the groundwork for quantitative evaluation. It loads a golden benchmark dataset, converting it into `DeepEval Golden` objects, and splits it into training and testing sets. Critical evaluation criteria for 'Answer Relevance', 'Faithfulness', and 'Answer Completeness' are defined, and `DeepEval GEval` metrics are initialized to objectively assess the RAG system's performance.

Let's test the end-to-end RAG pipeline with a sample financial question.

In [ ]:
# Sample financial question
sample_question = "What risks did Tesla highlight in its latest SEC filing?"

print(f"Asking question: '{sample_question}'\n")

# Get the answer using the RAG pipeline
answer, retrieved_documents, formatted_context = get_rag_answer(sample_question)

print("--- Retrieved Context ---")
print(formatted_context)

print("\n--- Generated Answer ---")
print(answer)

print("\n--- Raw Retrieved Documents (for inspection) ---")
# Display only a summary of the retrieved documents to avoid verbose output
for i, doc in enumerate(retrieved_documents):
    print(f"Document {i+1}: Source={os.path.basename(doc.metadata.get('source', 'N/A'))}, Content Snippet='{doc.page_content[:150]}...'\n")

Asking question: 'What risks did Tesla highlight in its latest SEC filing?'

--- Retrieved Context ---
-- Document 1 --
Source: sec_filings_10k.txt
Content: Reports on Form 8-K, proxy statements and other information with the SEC. In addition, the SEC maintains a website ( www.sec.gov ) that contains reports, proxy and information statements, and other information regarding issuers that file electronically. Our website is located at www.tesla.com , and our reports, amendments thereto, proxy statements and other information are also made available, free of charge, on our investor relations website at ir.tesla.com as soon as reasonably practicable after we electronically file or furnish such information with the SEC. The information posted on our website is not incorporated by reference into this Annual Report on Form 10-K. ITEM 1A. RISK FACTORS You should carefully consider the risks described below together with the other information set forth in this report, which could materially aff

## 1.5 Load the gold benchmark dataset and evaluate the baseline prompt


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from deepeval.dataset import Golden

# 1. Load the CSV benchmark dataset
try:
    benchmark_df = pd.read_csv('/content/golden_benchmark_dataset.csv')
    print(f"Successfully loaded {len(benchmark_df)} examples from golden_benchmark_dataset.csv")
except FileNotFoundError:
    print("Error: golden_benchmark_dataset.csv not found. Please ensure it's in the /content/ directory.")
    benchmark_df = pd.DataFrame() # Create an empty DataFrame to prevent further errors

# 2. Validate that all required columns are present
required_columns = ['question', 'response', 'context', 'supporting_sources']
if not all(col in benchmark_df.columns for col in required_columns):
    print(f"Error: Missing one or more required columns in the benchmark dataset. Expected: {required_columns}")
    print(f"Found: {benchmark_df.columns.tolist()}")
    benchmark_df = pd.DataFrame() # Clear DataFrame if columns are missing

if not benchmark_df.empty:
    print("Benchmark dataset head:")
    display(benchmark_df.head())


Successfully loaded 20 examples from golden_benchmark_dataset.csv
Benchmark dataset head:


,question,response,source_hint,supporting_sources,context
0,Given the internal control audit for the fisca...,No material weaknesses were identified. Ernst ...,Apple,sec_filings.txt,"[Source 1] reporting as of September&#160;27, ..."
1,There is market chatter about executive transi...,The Principal Executive Officer is Timothy D. ...,Apple,sec_filings.txt,[Source 1] and in the capacities and on the da...
2,Does Apple’s 10-K disclose any recent changes ...,No. Item 9A of the filing states there were no...,Apple,sec_filings.txt,[Source 1] changes in the Company&#8217;s inte...
3,A client is skeptical of automated financial r...,Apple states that internal controls are design...,Apple,sec_filings.txt,[Source 1] statements in accordance with gener...
4,"Based on the 2025 Form 10-K, what was the as-o...",Apple assessed internal control over financial...,Apple,sec_filings.txt,"[Source 1] reporting as of September&#160;27, ..."


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from deepeval.dataset import Golden

# 3. Convert each row into a structured dictionary format and 4. Create DeepEval Golden objects
golden_examples = []
if not benchmark_df.empty:
    for index, row in benchmark_df.iterrows():
        golden = Golden(
            input=row['question'],
            # Ensure expected_output is a non-empty string; provide a placeholder if it's empty
            expected_output=str(row['response']) if pd.notna(row['response']) and str(row['response']).strip() != '' else 'N/A_EXPECTED_RESPONSE',
            retrieval_context=[str(row['context'])],
            # supporting_context=row['supporting_sources'] # Use this if supporting_sources is a list of strings
            # For now, we'll omit supporting_sources as it's not directly used in initial RAG eval
        )
        golden_examples.append(golden)
    print(f"Created {len(golden_examples)} DeepEval Golden objects.")
else:
    print("Cannot create Golden objects as the benchmark DataFrame is empty or invalid.")


# 5. Split the benchmark into train and test sets
if golden_examples:
    # Using a fixed random_state for reproducibility
    train_set, test_set = train_test_split(golden_examples, test_size=0.3, random_state=42)
    # 6. Print the number of examples in each split for verification
    print(f"Train set size: {len(train_set)}")
    print(f"Test set size: {len(test_set)}")
else:
    train_set = []
    test_set = []
    print("Train and test sets are empty.")

Created 20 DeepEval Golden objects.
Train set size: 14
Test set size: 6


Now that we have the `test_set` of `Golden` examples, we can proceed to evaluate the baseline RAG prompt.

## 1.6 Define the evaluation metrics

In [ ]:
# Define the evaluation criteria.

RELEVANCE_CRITERIA = (
    "Evaluate whether the generated answer directly addresses the broker’s financial question and stays aligned with the expected financial reasoning."
    "Reward answers that correctly reference market movements, company‑specific events, risk factors, sentiment signals, or disclosures relevant to the question."
    "Penalize answers that introduce unrelated market commentary, misinterpret the financial context, or omit key company‑ or market‑specific details."
)

FAITHFULNESS_CRITERIA = (
    "Evaluate whether the answer remains grounded in the retrieved financial evidence."
    "Reward answers that clearly rely on the retrieved news articles, stock‑price data, SEC filings, analyst commentary, or regulatory disclosures."
    "Penalize hallucinated financial claims, unsupported interpretations, or statements that cannot be traced back to the retrieved documents."
)

COMPLETENESS_CRITERIA = (
    "Evaluate whether the answer fully resolves the broker’s question using all relevant retrieved context."
    "Reward answers that incorporate the necessary market signals, price trends, filing disclosures, risk factors, or event‑driven insights required to give a complete financial explanation."
    "Penalize answers that are partially correct, miss important disclosures, ignore key market movements, or fail to mention critical risks or catalysts."
)

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

answer_relevance_metric = GEval(
    name="Answer Relevance",
    criteria=RELEVANCE_CRITERIA,
    evaluation_params=[
        SingleTurnParams("input"),
        SingleTurnParams("actual_output"),
        SingleTurnParams("expected_output"),
    ],
    model=ANSWER_GENERATION_MODEL
)

faithfulness_metric = GEval(
    name="Faithfulness",
    criteria=FAITHFULNESS_CRITERIA,
    evaluation_params=[
        SingleTurnParams("input"),
        SingleTurnParams("actual_output"),
        SingleTurnParams("retrieval_context"),
    ],
    model=ANSWER_GENERATION_MODEL
)

answer_completeness_metric = GEval(
    name="Answer Completeness",
    criteria=COMPLETENESS_CRITERIA,
    evaluation_params=[
        SingleTurnParams("input"),
        SingleTurnParams("actual_output"),
        SingleTurnParams("expected_output"),
    ],
    model=ANSWER_GENERATION_MODEL
)

evaluation_metrics = [
    answer_relevance_metric,
    faithfulness_metric,
    answer_completeness_metric,
]

print("DeepEval 4.0.2 metrics initialized successfully.")


DeepEval 4.0.2 metrics initialized successfully.


## 1.7 Evaluate the baseline RAG on the golden examples dataset

### Note:
This cell defines the `evaluate_rag_pipeline` function, a reusable utility for evaluating any RAG configuration against a given `test_set` of `Golden` examples. It iterates through each golden example, generates an answer using the provided `rag_answer_function`, and constructs a `DeepEval LLMTestCase`. It then measures each `evaluation_metrics` against the `test_case`, ensuring that `retrieval_context` is properly handled. The function collects individual metric scores and computes their averages, providing a comprehensive summary of the RAG pipeline's performance.

In [ ]:
import numpy as np

def evaluate_rag_pipeline(
    rag_answer_function,
    dataset_name: str,
    test_set,
    metrics,
    display_results: bool = True
):
    print(f"\n--- Starting Evaluation for {dataset_name} RAG Pipeline ---")
    all_metric_scores = {metric.name: [] for metric in metrics}

    for i, golden in enumerate(test_set):
        print(f"\nEvaluating example {i+1}/{len(test_set)}...")

        generated_answer, retrieved_docs, formatted_context = rag_answer_function(golden.input)

        retrieval_context_contents = [doc.page_content for doc in retrieved_docs]

        # Ensure retrieval_context_contents is not empty for metrics that require it
        if not retrieval_context_contents:
            retrieval_context_contents = ["No retrieval context found for faithfulness evaluation."]

        test_case = LLMTestCase(
            input=golden.input,
            actual_output=generated_answer,
            expected_output=golden.expected_output,
            retrieval_context=retrieval_context_contents
        )

        evaluation_result = evaluate([test_case], metrics=metrics)

        # DeepEval 4.0.2: metric results live in metrics_data
        for metric_data in evaluation_result.test_results[0].metrics_data:
            metric_name = metric_data.name.replace(" [GEval]", "")
            all_metric_scores[metric_name].append(metric_data.score)

            if display_results:
                print(f"  {metric_name}: {metric_data.score}")

    average_scores = {
        name: np.mean(scores)
        for name, scores in all_metric_scores.items()
        if scores
    }

    if display_results:
        print(f"\n--- {dataset_name} Evaluation Summary ---")
        for name, avg_score in average_scores.items():
            print(f"Average {name}: {avg_score:.2f}")

    print(f"--- Finished Evaluation for {dataset_name} RAG Pipeline ---")
    return average_scores

Now, let's use this function to evaluate our `get_rag_answer` baseline pipeline on the `test_set` using the `evaluation_metrics` we defined earlier.

In [ ]:
from deepeval import evaluate

# Evaluate the baseline RAG pipeline
baseline_avg_scores = evaluate_rag_pipeline(
    rag_answer_function=get_rag_answer,
    dataset_name="Baseline",
    test_set=test_set,
    metrics=evaluation_metrics,
    display_results=True
)


--- Starting Evaluation for Baseline RAG Pipeline ---

Evaluating example 1/6...


✨ You're running DeepEval's latest Answer Relevance [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Completeness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()

...

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                            ┃ Average Score            ┃ Pass Rate         ┃ Total       │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━ │
│  Answer Relevance [GEval]                          │ 0.76                     │ 100.00%           │ 1           │
│  Faithfulness [GEval]                              │ 0.95                     │ 100.00%           │ 1           │
│  Answer Completeness [GEval]                       │ 0.67                     │ 100.00%           │ 1           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=157553;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 3.3s | token cost: 0.0007664999999999999 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

  Answer Relevance: 0.7566217653610435
  Faithfulness: 0.9501047931303844
  Answer Completeness: 0.6681428541348999

--- Baseline Evaluation Summary ---
Average Answer Relevance: 0.65
Average Faithfulness: 0.84
Average Answer Completeness: 0.61
--- Finished Evaluation for Baseline RAG Pipeline ---


# Section 2: DeepEval Prompt Optimization Layer

### DeepEval Prompt Optimization Setup
This phase prepares the DeepEval environment for prompt optimization using the GEPA (Genetic-Pareto Prompt Optimization) algorithm. It defines the initial prompt template by combining the baseline system and user prompts, making it compatible with DeepEval's `Prompt` object. Additionally, it configures two separate language models: one (`optim_model`) for suggesting diverse prompt mutations during optimization and another (`answer_llm`) for generating answers during the evaluation phase, ensuring both creativity in search and consistency in testing. A `model_callback` function is also defined to bridge DeepEval's optimization loop with the LLM's answer generation.

## 2.1 Define the prompt template and model callback


In [ ]:
# DeepEval prompt optimizer uses a Prompt object as the starting point.
PROMPT_TEMPLATE_V1 = Prompt(
    text_template=(
        BASELINE_SYSTEM_PROMPT
        + "\n\n"
        + BASELINE_USER_PROMPT
            .replace("{question}", "{input}")
            .replace("{context}", "{context}")
    )
)

### Note:
This cell initializes the `PROMPT_TEMPLATE_V1` object, which represents the starting prompt for DeepEval's optimization process. It combines the `BASELINE_SYSTEM_PROMPT` and `BASELINE_USER_PROMPT` into a single template, replacing `{question}` with `{input}` to align with DeepEval's `Prompt` object expected parameters. This `Prompt` object is the foundation upon which GEPA will iteratively generate and evaluate mutated prompt versions to improve performance.

In [ ]:
# Model used during prompt optimization to suggest prompt mutations.
# NOTE:
# A slightly higher temperature helps encourage prompt mutation diversity.

optim_model = GPTModel(
    model=ANSWER_GENERATION_MODEL,
    temperature=0.5,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

# LLM used to generate answers during prompt evaluation.

answer_llm = ChatOpenAI(
    model=ANSWER_GENERATION_MODEL,
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

In [ ]:
def model_callback(prompt: Prompt, golden: Golden) -> str:
    """
    Called by DeepEval during prompt optimization.

    The function:
    1. Injects the benchmark question and context into the prompt
    2. Sends the final prompt to the LLM
    3. Returns the generated answer
    """

    interpolated = prompt.interpolate(
        input=golden.input,
        context=golden.context[0] if golden.context else "",
    )

    response = answer_llm.invoke([
        SystemMessage(content=(
            "You are a financial intelligence assistant for brokers. "
            "Follow the user prompt exactly and answer only from the provided context."
        )),
        HumanMessage(content=interpolated),
    ])

    return response.content

### Note:
This cell configures the language models used specifically for prompt optimization. `optim_model` (a `GPTModel` with a `temperature` of 0.5) is used by GEPA to suggest diverse prompt mutations, encouraging exploration of different phrasing and instructions. `answer_llm` (a `ChatOpenAI` model with `temperature` 0) is used to generate answers during the evaluation phase of the optimization process. This separation ensures that the model suggesting changes is creative, while the model being evaluated provides consistent, deterministic outputs.

## 2.2 Optimize the prompt with GEPA

In [ ]:
# ── GEPA: Genetic-Pareto Prompt Optimisation ──────────────────────────────────
# GEPA iteratively mutates and evaluates prompts to improve performance
# across multiple evaluation metrics.

os.environ["DEEPEVAL_RETRY_MAX_ATTEMPTS"] = "2"
os.environ["DEEPEVAL_RETRY_INITIAL_SECONDS"] = "1.0"
os.environ["DEEPEVAL_RETRY_EXP_BASE"] = "1.0"
os.environ["DEEPEVAL_RETRY_JITTER"] = "1.0"
os.environ["DEEPEVAL_RETRY_CAP_SECONDS"] = "5.0"
os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "90"

### Note:
This cell configures environment variables to control DeepEval's retry logic and timeouts during the prompt optimization process. These settings (`DEEPEVAL_RETRY_MAX_ATTEMPTS`, `DEEPEVAL_RETRY_INITIAL_SECONDS`, `DEEPEVAL_RETRY_EXP_BASE`, `DEEPEVAL_RETRY_JITTER`, `DEEPEVAL_RETRY_CAP_SECONDS`, `DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE`) are crucial for handling transient API errors and ensuring that the optimization runs robustly without prematurely failing due to temporary network or API issues. This makes the optimization process more resilient.

In [ ]:
start_time = time.time()

# Configure the PromptOptimizer with the GEPA algorithm.
# - iterations: Number of evolutionary steps to perform.
# - pareto_size: Number of top-performing, diverse candidates to track.
# - minibatch_size: Number of golden examples to evaluate per iteration.
# - random_seed: Ensures reproducibility of the mutation/selection process.
# - tie_breaker: Strategy to use when a child prompt has the same score as its parent.

# Configure the PromptOptimizer using the GEPA algorithm.
gepa_optimizer = PromptOptimizer(
    algorithm=GEPA(
        iterations=4,
        pareto_size=2,     # hold 1 frontier candidates
        minibatch_size=4,  # sample 4 samples per evaluation
        random_seed=42,
        tie_breaker=TieBreaker.PREFER_CHILD,  # breaks ties in favour of the mutation
    ),
    model_callback=model_callback,  # The function that executes the prompt and returns output
    metrics=evaluation_metrics,       # The list of GEval metrics to optimize for
    optimizer_model=optim_model,    # The LLM that generates the optimized prompt variants
)

# Ensure the algorithm instance specifically uses the provided optimizer model.
gepa_optimizer.algorithm.optimizer_model = optim_model

# Start the optimization process.
# This will iteratively evaluate and mutate the 'PROMPT_TEMPLATE_V1'
# using the provided benchmark dataset.
gepa_prompt = gepa_optimizer.optimize(
    prompt=PROMPT_TEMPLATE_V1,
    goldens=train_set,
)

end_time = time.time()

print(f"Execution Time: {end_time - start_time:.4f} seconds")

Output()

                                          ✨ GEPA Evolutionary Mutations                                           
╭────┬───────────────┬──────────┬──────────┬───────────┬─────────────────────────────────────────────────┬────────╮
│  # │    Outcome    │   Before │    After │   Δ Score │ Note                                            │   Time │
├────┼───────────────┼──────────┼──────────┼───────────┼─────────────────────────────────────────────────┼────────┤
│  1 │  ✔ accepted   │   0.0015 │   0.1459 │   +0.1445 │ Accepted by Pareto non-domination               │ 74.44s │
├────┼───────────────┼──────────┼──────────┼───────────┼─────────────────────────────────────────────────┼────────┤
│  2 │   ↷ skipped   │   0.1459 │   0.2487 │   +0.1027 │ Skipped (minibatch score did not improve)       │ 50.81s │
├────┼───────────────┼──────────┼──────────┼───────────┼─────────────────────────────────────────────────┼────────┤
│  3 │   ↷ skipped   │   0.1459 │   0.2131 │   +0.0672 │ Skipped (minibatch score did not improve)       │ 51.86s │
├────┼───────────────┼──────────┼──────────┼───────────┼─────────────────────────────────────────────────┼────────┤
│  4 │   ↷ skipped   │   0.1459 │   0.2134 │   +0.0675 │ Skipped (minibatch score did not improve)       │ 46.06s │
╰────┴───────────────┴──────────┴──────────┴───────────┴─────────────────────────────────────────────────┴────────╯

                                               Final Pareto Archive                                                
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  Config ID                            Role         Scores                                              Aggregate  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 
  82177fdd… ★                          child        [0.201, 0.091]                                         0.1459  
 ───────────────────────────────────────────────────────────────────────────────────────────────────────────────── 

Execution Time: 223.1975 seconds


### GEPA Optimization Execution and Prompt Comparison
This section executes the GEPA (Genetic-Pareto Prompt Optimization) process, iteratively mutating and evaluating prompt variants against the training set to identify a more effective prompt. After the optimization, the original baseline prompt and the newly `GEPA Optimized Prompt` are printed side-by-side, allowing for a direct comparison of the changes introduced by the automated optimization. Following this, a new RAG function (`optimized_prompt_answer`) is defined, specifically incorporating this optimized prompt, and is then evaluated against the unseen test set to measure its performance improvement over the baseline.

In [ ]:
print("\nOriginal Prompt:")
print(PROMPT_TEMPLATE_V1.text_template)

print("\nGEPA Optimized Prompt:")
print(gepa_prompt.text_template)


Original Prompt:
You are a highly knowledgeable financial intelligence assistant.
Your goal is to answer questions based *only* on the provided financial context.
Do NOT use any outside knowledge. If the answer is not in the context, state that you cannot answer based on the provided information.
Maintain a professional and neutral tone. Prioritize factual accuracy.
Cite the source(s) from the retrieved documents if explicitly asked.

Broker Question:
{input}

Retrieved Context:
{context}

GEPA Optimized Prompt:
You are a highly knowledgeable financial intelligence assistant. Your goal is to answer questions based *only* on the provided financial context. Do NOT use any outside knowledge. If the answer is not in the context, **do not simply state you cannot answer**; instead, explain the relevant financial concepts based on the context. Always engage with the provided context to support your answers. Maintain a professional and neutral tone. Prioritize factual accuracy. Cite the sourc

## 2.3 Evaluate the optimized prompt on the benchmark dataset

In [ ]:
# -------------------------------------------------------------------
# Create a runtime function using the GEPA optimized prompt
# -------------------------------------------------------------------

def optimized_prompt_answer(question: str) -> dict:
    """
    Generate answers using:
    - the GEPA optimized prompt
    - the baseline retrieval pipeline
    """

    # Retrieve relevant chunks from the vector database
    docs = retrieve_docs(question, k=4)

    # Convert retrieved chunks into prompt-ready context
    context = format_docs(docs)

    # Inject runtime values into the optimized prompt
    final_prompt = gepa_prompt.interpolate(
        input=question,
        context=context,
    )

    # Generate the final response
    response = answer_llm.invoke([
        SystemMessage(content=(
            "You are a financial intelligence assistant for brokers. "
            "Answer only from the provided context."
        )),
        HumanMessage(content=final_prompt),
    ])

    # Return as a tuple: (answer_text, retrieved_docs, formatted_context_str)
    return response.content, docs, context

## 2.4 Compare optimized prompt with the baseline prompt


In [ ]:
# -------------------------------------------------------------------
# Evaluate the GEPA optimized prompt only on the TEST set
# -------------------------------------------------------------------

start_time = time.time()

print("Evaluating the GEPA optimized prompt on the TEST set...")

optimized_test_eval = evaluate_rag_pipeline(
    rag_answer_function=optimized_prompt_answer,
    dataset_name="GEPA",
    test_set=test_set,
    metrics=evaluation_metrics,
    display_results=True
)

end_time = time.time()

# Calculate overall mean from the individual metric scores
overall_mean_score = np.mean(list(optimized_test_eval.values()))

print(f"\nOptimized Test Overall Score: {overall_mean_score:.2f}")

print(f"\nExecution Time: {end_time - start_time:.2f} seconds")

Evaluating the GEPA optimized prompt on the TEST set...

--- Starting Evaluation for GEPA RAG Pipeline ---

Evaluating example 1/6...


✨ You're running DeepEval's latest Answer Relevance [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Faithfulness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Answer Completeness [GEval] Metric! (using gpt-4o-mini, strict=False, 
async_mode=True)...

Output()

...

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                            ┃ Average Score            ┃ Pass Rate         ┃ Total       │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━ │
│  Answer Relevance [GEval]                          │ 0.80                     │ 100.00%           │ 1           │
│  Faithfulness [GEval]                              │ 0.97                     │ 100.00%           │ 1           │
│  Answer Completeness [GEval]                       │ 0.76                     │ 100.00%           │ 1           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=342828;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 3.04s | token cost: 0.0008447999999999999 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

  Answer Relevance: 0.8012673507448081
  Faithfulness: 0.9731058584489498
  Answer Completeness: 0.7579268017932852

--- GEPA Evaluation Summary ---
Average Answer Relevance: 0.70
Average Faithfulness: 0.87
Average Answer Completeness: 0.63
--- Finished Evaluation for GEPA RAG Pipeline ---

Optimized Test Overall Score: 0.73

Execution Time: 46.83 seconds


# Section 3: RAG Configuration Tuning and Evaluation


### RAG Configuration Tuning and Evaluation Process
This part of the notebook defines multiple distinct RAG configurations, each varying key parameters such as the number of retrieved chunks (`k`), LLM temperature, top-p sampling, and maximum token limits. These configurations are designed to explore different retrieval and generation strategies. A runtime answering function (`answer_with_rag_config`) is created to allow for dynamic application of these configurations. A dedicated evaluation function (`evaluate_rag_configuration`) is then used to systematically run and measure the performance of each defined RAG configuration against the `test_set` using the `DeepEval` metrics.

## 3.1 Define the RAG configurations

In [ ]:
# Each configuration uses different retrieval and generation settings.

# We tune only runtime parameters:
# - k: number of retrieved chunks
# - temperature: randomness of generation
# - top_p: nucleus sampling
# - max_tokens: response length

RAG_CONFIGURATIONS = [
    {
        "name": "Config_1_Precision_Focused",
        "k": 3,
        "temperature": 0.0,
        "top_p": 0.9,
        "max_tokens": 400,
        "rationale": (
            "Tight retrieval and deterministic generation. "
            "Ideal for brokers needing crisp, fact‑aligned answers grounded in "
            "specific SEC filings, earnings disclosures, or single‑event news items."
        ),
    },
    {
        "name": "Config_2_Balanced",
        "k": 4,
        "temperature": 0.2,
        "top_p": 0.95,
        "max_tokens": 600,
        "rationale": (
            "Balanced retrieval depth and controlled creativity. "
            "Useful for general broker queries involving mixed signals—market news, "
            "price movements, analyst commentary, and company‑specific updates."
        ),
    },
    {
        "name": "Config_3_Context_Heavy",
        "k": 6,
        "temperature": 0.3,
        "top_p": 1.0,
        "max_tokens": 800,
        "rationale": (
            "Broader retrieval for multi‑document synthesis. "
            "Best for complex financial reasoning where brokers need cross‑source "
            "context—e.g., combining SEC risk factors, multi‑day news cycles, "
            "and price‑trend analysis into a single coherent answer."
        ),
    },
]

print("Defined RAG configurations:\n")
for config in RAG_CONFIGURATIONS:
    print(f"{config['name']}")
    print(f"Rationale: {config['rationale']}\n")


Defined RAG configurations:

Config_1_Precision_Focused
Rationale: Tight retrieval and deterministic generation. Ideal for brokers needing crisp, fact‑aligned answers grounded in specific SEC filings, earnings disclosures, or single‑event news items.

Config_2_Balanced
Rationale: Balanced retrieval depth and controlled creativity. Useful for general broker queries involving mixed signals—market news, price movements, analyst commentary, and company‑specific updates.

Config_3_Context_Heavy
Rationale: Broader retrieval for multi‑document synthesis. Best for complex financial reasoning where brokers need cross‑source context—e.g., combining SEC risk factors, multi‑day news cycles, and price‑trend analysis into a single coherent answer.



## 3.2 Create a runtime answering function


This function answers a question using the GEPA-optimized prompt and dynamic RAG configuration settings, including the number of retrieved chunks (`k`), LLM temperature, top-p sampling, and max tokens. This function is essential for systematically testing different RAG configurations.

In [ ]:
def answer_with_rag_config(question: str, config: dict) -> dict:
    """
    Answer a question using:
    - the fixed Chroma vector store,
    - the GEPA-optimized prompt,
    - and the runtime configuration settings.
    """

    # Retrieve the most relevant chunks using the current k value.
    docs = vector_store.similarity_search(question, k=config["k"])

    # Turn the retrieved chunks into a prompt-ready context block.
    context = format_docs(docs)

    # Inject the question and retrieved context into the optimized prompt.
    final_prompt = gepa_prompt.interpolate(
        input=question,
        context=context,
    )

    # Create the answer model using the current config settings.
    # top_p and max_tokens are passed through model_kwargs.
    config_llm = ChatOpenAI(
        model=ANSWER_GENERATION_MODEL,
        temperature=config["temperature"],
        model_kwargs={
            "top_p": config["top_p"],
            "max_tokens": config["max_tokens"],
        },
        api_key=os.getenv("OPENAI_API_KEY"),
        base_url=os.getenv("OPENAI_BASE_URL"),
    )

    # Ask the model to generate the final answer.
    response = config_llm.invoke([
        SystemMessage(content=(
            "You are a financial intelligence assistant for brokers."
            "Answer only from the provided context and stay grounded."
        )),
        HumanMessage(content=final_prompt),
    ])

    return {
        "answer": response.content,
        "context": context,
        "docs": docs,
    }

## 3.3 Evaluate one configuration on a benchmark set

### Note:
This cell defines the `evaluate_rag_configuration` function, a core utility for evaluating a single RAG configuration. It takes a `config` dictionary, a subset of `goldens` (typically the `test_set`), and a `label`. It iterates through each golden example, generates an answer using `answer_with_rag_config` with the current configuration, and creates a `DeepEval LLMTestCase`. It then measures all defined `evaluation_metrics` against the test case, ensuring proper handling of `retrieval_context`. Finally, it computes and returns the metric-wise average scores and an overall average score for the given configuration, along with its name and rationale.

In [ ]:
def evaluate_rag_configuration(config: dict, goldens_subset, label: str) -> dict:
    """
    Evaluate one RAG configuration on a given benchmark subset.

    Returns:
    - metric-wise average scores
    - overall average score
    """

    print("\n" + "=" * 90)
    print(f"Evaluating: {config['name']} | {label}")
    print("=" * 90)

    metric_scores = {
        metric.name: []
        for metric in evaluation_metrics
    }

    for idx, golden in enumerate(goldens_subset, start=1):
        print(f"\nEvaluating Example {idx}/{len(goldens_subset)}")

        # Generate answer using the current config
        result = answer_with_rag_config(
            question=golden.input,
            config=config,
        )

        # Extract page_content from each document for retrieval_context
        retrieval_context_contents = [doc.page_content for doc in result["docs"]]

        # Ensure retrieval_context_contents is not empty for metrics that require it
        if not retrieval_context_contents:
            retrieval_context_contents = ["No retrieval context found for faithfulness evaluation."]

        # Build the DeepEval test case
        test_case = LLMTestCase(
            input=golden.input,
            actual_output=result["answer"],
            expected_output=golden.expected_output,
            retrieval_context=retrieval_context_contents,
        )

        # Measure all metrics
        for metric in evaluation_metrics:
            try:
                metric.measure(test_case)

                if metric.score is not None:
                    metric_scores[metric.name].append(metric.score)

                print(f"{metric.name}: {metric.score}")

            except Exception as e:
                print(f"{metric.name} failed: {e}")

    # Compute average scores for each metric
    metric_means = {
        name: round(sum(scores) / len(scores), 3) if scores else 0.0
        for name, scores in metric_scores.items()
    }

    # Overall score is the mean of all metric means
    overall_score = round(
        sum(metric_means.values()) / len(metric_means),
        3
    ) if metric_means else 0.0

    print("\nMetric Means:")
    print(metric_means)
    print(f"Overall Score: {overall_score}")

    return {
        "configuration": config["name"],
        "metric_means": metric_means,
        "overall_score": overall_score,
        "rationale": config["rationale"],
    }

## 3.4 Run all configurations on the test set

### Note:
This cell runs all predefined `RAG_CONFIGURATIONS` on the `test_set`. It iterates through each configuration, calls the `evaluate_rag_configuration` function, and stores the results in the `configuration_results` list. This systematic evaluation allows for a quantitative comparison of how different retrieval and generation parameters impact the RAG system's performance across the DeepEval metrics. The execution time for evaluating all configurations is also measured and printed.

In [ ]:
configuration_results = []

start_time = time.time()

for config in RAG_CONFIGURATIONS:
    result = evaluate_rag_configuration(
        config=config,
        goldens_subset=test_set,
        label="Test Set",
    )
    configuration_results.append(result)

end_time = time.time()

print("\nCompleted evaluation on the test set.")
print(f"Execution Time: {end_time - start_time:.2f} seconds")


Evaluating: Config_1_Precision_Focused | Test Set

Evaluating Example 1/6


/tmp/ipykernel_3955/4067920859.py:6: UserWarning: Parameters {'max_tokens', 'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  result = evaluate_rag_configuration(


Output()

Output()

Answer Relevance: 0.8320821300824607


...

Output()

Answer Relevance: 0.7876717639713178


Output()

Faithfulness: 0.9851952796329894


Answer Completeness: 0.7309861056039113

Metric Means:
{'Answer Relevance': 0.695, 'Faithfulness': 0.882, 'Answer Completeness': 0.646}
Overall Score: 0.741

Completed evaluation on the test set.
Execution Time: 145.61 seconds


## 3.5 Compare configurations

### Comparing and Selecting the Best RAG Approach
After evaluating all defined RAG configurations, this section compiles all results, including the baseline and GEPA-optimized prompt evaluations, into a single comparison table. This table is then sorted by 'Overall Score' to clearly identify the top-performing RAG approach. The details of this `best_approach` are extracted and printed, highlighting its key metrics and rationale. Finally, the complete comparison data is saved as a CSV and a JSON file within the DeepEval artifacts directory, ensuring all evaluation results are persistently stored for future reference and analysis.

In [ ]:
# Add baseline and optimized prompt results so all five approaches are compared together.
all_results = [
    {
        "Approach": "Baseline Prompt",
        "Relevance": baseline_avg_scores.get("Answer Relevance", 0.0),
        "Faithfulness": baseline_avg_scores.get("Faithfulness", 0.0),
        "Completeness": baseline_avg_scores.get("Answer Completeness", 0.0),
        "Overall Score": sum(baseline_avg_scores.values()) / len(baseline_avg_scores) if baseline_avg_scores else 0.0,
        "Rationale": "Baseline prompt from Section 1.",
    },
    {
        "Approach": "GEPA Optimized Prompt",
        "Relevance": optimized_test_eval.get("Answer Relevance", 0.0),
        "Faithfulness": optimized_test_eval.get("Faithfulness", 0.0),
        "Completeness": optimized_test_eval.get("Answer Completeness", 0.0),
        "Overall Score": sum(optimized_test_eval.values()) / len(optimized_test_eval) if optimized_test_eval else 0.0,
        "Rationale": "GEPA-optimized prompt from Section 2.",
    },
]

# Add the three runtime RAG configurations.
for result in configuration_results:
    all_results.append({
        "Approach": result["configuration"],
        "Relevance": result["metric_means"].get("Answer Relevance", 0.0),
        "Faithfulness": result["metric_means"].get("Faithfulness", 0.0),
        "Completeness": result["metric_means"].get("Answer Completeness", 0.0),
        "Overall Score": result["overall_score"],
        "Rationale": result["rationale"],
    })

results_df = pd.DataFrame(all_results)

results_df = results_df.sort_values(
    by="Overall Score",
    ascending=False
).reset_index(drop=True)

print("\nRAG Configuration Comparison on Test Set:")
display(results_df)


RAG Configuration Comparison on Test Set:


,Approach,Relevance,Faithfulness,Completeness,Overall Score,Rationale
0,Config_3_Context_Heavy,0.695000,0.882000,0.646000,0.741000,Broader retrieval for multi‑document synthesis...
1,GEPA Optimized Prompt,0.697573,0.866741,0.634626,0.732980,GEPA-optimized prompt from Section 2.
2,Config_1_Precision_Focused,0.677000,0.856000,0.629000,0.721000,Tight retrieval and deterministic generation. ...
3,Config_2_Balanced,0.683000,0.850000,0.622000,0.718000,Balanced retrieval depth and controlled creati...
4,Baseline Prompt,0.646315,0.838237,0.606842,0.697131,Baseline prompt from Section 1.


## 3.6 Select the best overall approach

In [ ]:
# Select the best performing approach based on the 'Overall Score'
best_approach = results_df.iloc[0]

# Extract and print the details of the best approach
print("\n--- Best Performing RAG Approach ---")
print(f"Approach: {best_approach['Approach']}")
print(f"Relevance: {best_approach['Relevance']:.2f}")
print(f"Faithfulness: {best_approach['Faithfulness']:.2f}")
print(f"Completeness: {best_approach['Completeness']:.2f}")
print(f"Overall Score: {best_approach['Overall Score']:.2f}")
print(f"Rationale: {best_approach['Rationale']}")


--- Best Performing RAG Approach ---
Approach: Config_3_Context_Heavy
Relevance: 0.69
Faithfulness: 0.88
Completeness: 0.65
Overall Score: 0.74
Rationale: Broader retrieval for multi‑document synthesis. Best for complex financial reasoning where brokers need cross‑source context—e.g., combining SEC risk factors, multi‑day news cycles, and price‑trend analysis into a single coherent answer.


## 3.7 Save the tuning results


In [ ]:
import json

# Define file paths for saving results
comparison_csv_path = os.path.join(DEEPEVAL_ARTIFACTS_PATH, "rag_comparison_results.csv")
full_results_json_path = os.path.join(DEEPEVAL_ARTIFACTS_PATH, "rag_all_evaluation_results.json")

# 1. Save the full configuration comparison dataframe as a CSV file
results_df.to_csv(comparison_csv_path, index=False)
print(f"RAG configuration comparison saved to: {comparison_csv_path}")

# 2. Save all evaluation results as a JSON file
# Convert numpy types to native Python types for JSON serialization
serializable_all_results = []
for item in all_results:
    serializable_item = {}
    for k, v in item.items():
        if isinstance(v, (np.float32, np.float64, np.int32, np.int64)):
            serializable_item[k] = v.item() # Convert numpy scalar to Python scalar
        else:
            serializable_item[k] = v
    serializable_all_results.append(serializable_item)

with open(full_results_json_path, 'w') as f:
    json.dump(serializable_all_results, f, indent=4)
print(f"All evaluation results (including best approach) saved to: {full_results_json_path}")

RAG configuration comparison saved to: ./deepeval_artifacts/rag_comparison_results.csv
All evaluation results (including best approach) saved to: ./deepeval_artifacts/rag_all_evaluation_results.json


# Section 4: Final RAG Inference Pipeline with the Optimized Prompt and Best RAG Configuration

### Final RAG Inference Pipeline
This section introduces the `final_rag_inference` function, which acts as the entry point for the optimized RAG system. This helper function intelligently selects and utilizes the best-performing RAG approach—whether it's the baseline, the GEPA-optimized prompt, or one of the fine-tuned configurations—based on the results of the comprehensive evaluation. It ensures that all subsequent queries are processed using the most effective RAG strategy identified. A set of realistic `final_test_cases` are also defined, representing complex broker questions, to thoroughly demonstrate the end-to-end functionality of the fully optimized system.

## 4.1 Define the final inference helper


In [ ]:
import numpy as np

def find_rag_config_by_name(config_name: str, config_list: list) -> dict:
    """
    Helper function to find a full RAG configuration dictionary by its name.
    """
    for config_item in config_list:
        if config_item['name'] == config_name:
            return config_item
    return None

def final_rag_inference(question: str) -> dict:
    """
    Performs RAG inference using the best-performing approach determined during evaluation.
    Automatically selects between baseline, GEPA optimized, or tuned RAG configurations.

    Args:
        question (str): The user's question.

    Returns:
        dict: A dictionary containing the 'answer', 'docs' (retrieved documents),
              and 'context' (formatted context string).
    """
    # Use the global best_approach and RAG_CONFIGURATIONS determined in earlier steps
    global best_approach, RAG_CONFIGURATIONS

    approach_name = best_approach['Approach']
    result = {"answer": "", "docs": [], "context": ""}

    print(f"Using best approach: {approach_name}")

    if approach_name == "Baseline Prompt":
        # Baseline prompt uses the default k=4 in get_rag_answer
        answer, docs, context = get_rag_answer(question)
        result["answer"] = answer
        result["docs"] = docs
        result["context"] = context
    elif approach_name == "GEPA Optimized Prompt":
        answer, docs, context = optimized_prompt_answer(question)
        result["answer"] = answer
        result["docs"] = docs
        result["context"] = context
    else:
        # This is one of the tuned RAG configurations (e.g., Config_3_Context_Heavy)
        config_details = find_rag_config_by_name(approach_name, RAG_CONFIGURATIONS)
        if config_details:
            rag_config_result = answer_with_rag_config(question, config_details)
            result["answer"] = rag_config_result["answer"]
            result["docs"] = rag_config_result["docs"]
            result["context"] = rag_config_result["context"]
        else:
            print(f"Error: Unknown RAG configuration '{approach_name}'. Cannot perform inference.")

    return result

print("Final RAG inference function 'final_rag_inference' defined.")

Final RAG inference function 'final_rag_inference' defined.


## 4.2 Define the final test cases

In [ ]:
final_test_cases = [
    {
        "id": 1,
        "question": (
            "Compare the total revenue and net income reported by Apple, Amazon, "
            "and Alphabet in their latest 10-Q filings. Which company grew fastest "
            "year-over-year?"
        )
    },
    {
        "id": 2,
        "question": (
            "Based on this week's news sentiment around Amazon, would you classify "
            "the current signal as bullish or bearish, and does the price data support that?"
        )
    },
    {
        "id": 3,
        "question": (
            "Should I invest in Apple right now? I heard Tim Cook is stepping down — "
            "is that a red flag? How's the stock been doing, and is there anything I should "
            "be worried about with the company's financials or risks?"
        )
    },
    {
        "id": 4,
        "question": (
            "Is now a good time to buy Bitcoin? News is saying it just crossed $78k — "
            "am I too late? What's the price trend looked like over the last few months, "
            "and what's driving the current rally?"
        )
    },
    {
        "id": 5,
        "question": (
            "I want to invest in AI. Out of Google, Amazon, and Apple, which one looks "
            "like the best bet right now? How are they performing, what are they saying "
            "about their AI business, and what's the buzz around them?"
        )
    },
]

## 4.3 Run the final test cases

In [ ]:
final_inference_results = []

for test_case in final_test_cases:
    question = test_case["question"]

    print("\n" + "=" * 100)
    print(f"Test Case ID: {test_case['id']}")
    print(f"Question: {question}")

    # Generate answer using the final inference function
    inference_output = final_rag_inference(question)

    answer = inference_output["answer"]
    retrieved_docs = inference_output["docs"]

    # Extract unique source files
    source_files = set()
    for doc in retrieved_docs:
        source = doc.metadata.get('source', 'N/A')
        if source != 'N/A':
            source_files.add(os.path.basename(source))

    # Get the name of the best approach
    selected_approach = best_approach['Approach']

    print(f"\nSelected Approach: {selected_approach}")
    print(f"Generated Answer: {answer}")
    print(f"Source Files Used: {', '.join(source_files) if source_files else 'None'}")

    # Store the results
    final_inference_results.append({
        "id": test_case["id"],
        "question": question,
        "selected_approach": selected_approach,
        "generated_answer": answer,
        "source_files": list(source_files),
        "retrieved_documents": [{ "content": doc.page_content, "metadata": doc.metadata } for doc in retrieved_docs]
    })

print("\n--- Final Inference Run Completed ---")
# Optional: Display the first result for verification
# if final_inference_results:
#     print("\nFirst Final Inference Result (for inspection):")
#     display(final_inference_results[0])


Test Case ID: 1
Question: Compare the total revenue and net income reported by Apple, Amazon, and Alphabet in their latest 10-Q filings. Which company grew fastest year-over-year?
Using best approach: Config_3_Context_Heavy


/tmp/ipykernel_3955/4126578780.py:11: UserWarning: Parameters {'max_tokens', 'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  inference_output = final_rag_inference(question)



Selected Approach: Config_3_Context_Heavy
Generated Answer: Based on the retrieved context, we do not have specific figures for total revenue and net income for Apple, Amazon, and Alphabet from their latest 10-Q filings. However, we can analyze the growth trends mentioned in the context.

1. **Alphabet**: The context indicates that Alphabet's revenues increased by $15.5 billion from 2024 to 2025, primarily driven by growth in Google Cloud Platform, which suggests a significant year-over-year growth.

2. **Apple**: The context mentions increases in net sales across various product categories, including iPhones, iPads, and services, compared to the previous years. However, specific revenue growth figures are not provided.

3. **Amazon**: There is no specific information provided about Amazon's revenue or net income in the retrieved context.

To determine which company grew fastest year-over-year, we would need specific revenue and net income figures from their latest 10-Q filings. The c

/tmp/ipykernel_3955/4126578780.py:11: UserWarning: Parameters {'max_tokens', 'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  inference_output = final_rag_inference(question)



Selected Approach: Config_3_Context_Heavy
Generated Answer: Based on the provided context, there is no specific news sentiment around Amazon that indicates whether the current signal is bullish or bearish. The context primarily includes historical price data for Amazon (AMZN) and some information about Bitcoin, but it does not provide any recent news sentiment or analysis regarding Amazon's stock performance.

To assess whether the price data supports a bullish or bearish signal for Amazon, we can look at the historical price trends:

- On January 2, 2026, Amazon opened at $231.34 and closed at $226.50.
- On January 8, 2026, it opened at $243.06 and closed at $246.29, showing a positive movement.
- On January 13, 2026, it opened at $246.53 and closed at $242.60, indicating a slight decline.
- On January 29, 2026, it opened at $242.82 and closed at $241.73, which also reflects a downward trend.

From this data, we can observe that while there was a brief increase in price from January 

...


Selected Approach: Config_3_Context_Heavy
Generated Answer: Investing in Apple at this moment involves considering several factors, particularly the recent announcement of Tim Cook stepping down as CEO. While such leadership changes can often raise concerns among investors, the context suggests that this transition may not necessarily be a red flag.

1. **CEO Transition**: Tim Cook's departure is set for September, and he will be succeeded by John Ternus, Apple's top hardware engineer. This change comes after Apple's 50th anniversary and is seen by some analysts as a signal of strength rather than uncertainty. The new leadership could bring fresh perspectives, especially in areas like artificial intelligence, which is crucial for Apple's future competitiveness (Document 4, Document 6).

2. **Market Reaction**: Following the announcement of Cook's exit, Apple stock experienced a slight decline in after-hours trading, indicating a cautious market reaction (Document 6). However, the broa

/tmp/ipykernel_3955/4126578780.py:11: UserWarning: Parameters {'max_tokens', 'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  inference_output = final_rag_inference(question)



Selected Approach: Config_3_Context_Heavy
Generated Answer: Based on the retrieved context, Bitcoin has recently crossed the $78,000 mark, with reports indicating it touched approximately $78,300 before pulling back slightly. Over the past few months, Bitcoin's price has shown significant movement, particularly influenced by factors such as strong ETF inflows and geopolitical tensions, which have driven investor interest.

1. **Price Trend**: Bitcoin has been trading in a range around $75,000 to $78,300 recently. It climbed above $75,000 on April 17, 2026, and has seen a steady increase, reaching around $78,100 shortly after. The price has been supported by strong buying interest and a surge in trading volume, indicating a robust market sentiment.

2. **Current Rally Drivers**: The current rally appears to be driven by several factors:
   - **Geopolitical Tensions**: Investors are seeking Bitcoin as a hedge against instability, particularly in the Middle East.
   - **ETF Inflows**: Th

/tmp/ipykernel_3955/4126578780.py:11: UserWarning: Parameters {'max_tokens', 'top_p'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  inference_output = final_rag_inference(question)



Selected Approach: Config_3_Context_Heavy
Generated Answer: Based on the provided context, here's an overview of how Google (Alphabet Inc.) is positioning itself in the AI sector, which may help you in your investment decision-making regarding AI:

1. **Investment in AI**: Google has invested over $200 billion in research and development over the last five years, emphasizing their commitment to AI as a transformative platform. They view AI as a significant opportunity to drive productivity, reduce costs, and unlock new growth engines.

2. **AI Strategy**: Google aims to build the most advanced, safe, and responsible AI through a comprehensive approach that includes AI-optimized infrastructure, robust research teams, and a wide range of products that reach billions of users. Their AI infrastructure is designed to enhance their existing products like Search and YouTube, making them more efficient and effective.

3. **Innovation and Product Development**: Google is focused on continually

# Conclusion

## Actionable Insights:

- **GEPA-Optimized Prompt Effectiveness**: The GEPA (Genetic-Pareto Prompt Optimization) process successfully refined the system prompt, leading to a noticeable improvement in answer quality across metrics like Relevance, Faithfulness, and Completeness compared to the baseline prompt. This highlights the value of automated prompt engineering for domain-specific RAG systems.
- **Importance of Context Retrieval Depth**: The `Config_3_Context_Heavy` RAG configuration, which prioritizes a broader retrieval (`k=6`), emerged as the best-performing approach. This indicates that for complex financial intelligence queries, providing the LLM with a more extensive context from diverse sources (SEC filings, news, stock data) is crucial for generating comprehensive and grounded answers.
- **Multi-Source Grounding**: The RAG system effectively integrates and synthesizes information from various financial data sources (SEC filings, global news, stock prices). This capability directly addresses the initial problem of information overload for brokers, allowing the system to provide answers backed by evidence from multiple document types.
- **DeepEval for Objective Evaluation**: The consistent application of DeepEval metrics throughout the prompt optimization and RAG configuration tuning phases provided an objective and quantifiable way to compare different approaches. This methodical evaluation process was essential in identifying the most effective RAG strategy for financial intelligence.

## Recommendations:

- **Integrate into Broker Workflows**: Deploy the final RAG system with the optimized prompt and best-performing configuration (`Config_3_Context_Heavy`) directly into the broker's daily workflow. This could be a web interface or an API integration within their existing CRM/dashboard, allowing them to quickly query and retrieve grounded answers.
- **Continuous Prompt Refinement**: Establish a process for ongoing prompt optimization using DeepEval and GEPA. As market conditions evolve and new types of queries emerge, regularly update the benchmark dataset and re-run the optimization to maintain high answer quality.
- **Expand Data Sources**: Continuously integrate new relevant financial data sources, such as real-time market data feeds, analyst reports, and more diverse news sources. This will broaden the system's knowledge base and enhance its ability to provide comprehensive insights.
- **Feedback Loop Implementation**: Develop a feedback mechanism where brokers can rate the quality of answers or flag incorrect/unhelpful responses. This human feedback can be used to further fine-tune the RAG system and improve the golden benchmark dataset over time.
- **Performance Monitoring**: Implement robust monitoring for key RAG metrics (relevance, faithfulness, completeness) in a production environment. This will help detect any degradation in performance and trigger re-evaluation or re-training of the system as needed.

<font size=6>Power Ahead!</font>
___